# 10. 點過程：預報模型的共同語言

{doc}`第 9 章 <09_forecasting_intro>`把地震預報的目標訂了下來：不問「下一場
大地震什麼時候來」，改問「這個區域、這段時間內，發生某規模以上地震的機率
是多少」。問題換了，工具也得跟著換——機率要算得出來，就得先有一個能生出
機率的數學物件。

這一章要介紹的就是那個物件：**點過程**（point process）。它不是一個地震
模型，而是一種**語言**。第二部後面每一個模型——ETAS、應力釋放、EEPAS、
PPE，乃至於作業化系統裡真正在跑的引擎——都是用同一套語法寫成的句子。學會
這套語法之後，你讀任何一篇預報論文都能一眼看出它在做什麼：它的條件強度長
什麼樣？概似怎麼算？怎麼檢驗？這三個問題的答案就是一個模型的全部。

所以這一章刻意寫得比較「數學」：五條核心式子會從頭推到尾，不跳步。一句話
總結這章的成果——**只要你寫得下 $\lambda^*(t)$，機率、概似、模擬、檢驗就會
自動跟著出現**。

In [ ]:
from gdms_toolkit.viz import setup_plotly
setup_plotly()

## 10.1 從計數過程到條件強度

一份地震目錄剝到最素樸的樣子，就是時間軸上的一堆點 $t_1 < t_2 < \cdots$，
每個點掛著位置與規模。數學上用**計數過程**（counting process）描述它：
$N(t) = \#\{i : t_i \le t\}$，也就是「到時刻 $t$ 為止累積發生了幾個事件」。
它是一條階梯函數，每來一個地震就往上跳一階；你在{doc}`第 5 章 <05_seismic>`
畫過的累積事件數曲線就是它。真正有用的是這條階梯的**斜率**，但直接微分
沒有意義（幾乎處處為零、跳點處無窮），所以要先取期望值再微分——而且是在
**已知過去發生了什麼**的條件下取期望。這是整章最關鍵的一步。

把 $t$ 之前的全部事件記為 $H_t = \{(t_i,x_i,y_i,m_i): t_i < t\}$，稱為
**歷史**（history）。定義**條件強度函數**（conditional intensity）：

$$\lambda^*(t) \;=\; \lim_{\Delta \to 0}
  \frac{E\bigl[N(t+\Delta) - N(t) \,\big|\, H_t\bigr]}{\Delta}$$ (eq:cond-int)

星號是本書的固定記號，代表「條件於歷史」（10.8 節）。白話讀它：**在已經
知道過去所有地震的前提下，接下來這一瞬間，單位時間內平均會發生幾個地震**。

三個要留意的細節。第一，$\lambda^*$ 是**率**（rate）不是機率，單位是
「次／天」，可以大於 1、甚至大到 100；把它當機率讀是初學者的頭號錯誤。
第二，$\lambda^*$ 是**隨機的**（它依賴隨機的 $H_t$），但在兩個事件之間
歷史不會變，此時它退化成一條可以畫出來的確定性曲線——這個「分段確定性」
是下面所有推導的技術基礎。第三，這裡預設事件不會同時發生（**simple point
process**）；真實目錄偶爾出現時間完全相同的兩筆，那通常是資料處理的產物。

### 從 $\lambda^*$ 到「下一個地震什麼時候來」

條件強度看起來抽象，但它其實已經把整個過程的機率結構決定了。假設上一個
事件發生在 $t_{i-1}$，問下一個事件的時間 $T_i$ 落在哪裡。定義**存活函數**
$S(t) = P(T_i > t \mid H_{t_{i-1}})$，也就是「到 $t$ 為止還沒有下一個事件」
的機率。關鍵觀察：在 $(t_{i-1}, t)$ 內若還沒有新事件，歷史就沒有更新，
$\lambda^*$ 是一條已知的確定性函數，此時條件強度恰好等於存活分析裡的
**危害率** $\lambda^*(t) = f(t)/S(t)$，其中 $f$ 是 $T_i$ 的機率密度。
這一步值得停一秒：條件強度是「已知還沒發生，接下來瞬間發生的率」，危害率
是「已知存活到 $t$，瞬間死亡的率」——同一句話。又因 $f(t) = -S'(t)$，代入得

$$\lambda^*(t) \;=\; \frac{-S'(t)}{S(t)}
  \;=\; -\frac{\mathrm{d}}{\mathrm{d}t}\ln S(t)$$

兩邊從 $t_{i-1}$ 積到 $t$，用 $S(t_{i-1}) = 1$（剛發生完事件，當然還沒有
下一個）：

$$\begin{aligned}
-\int_{t_{i-1}}^{t} \lambda^*(u)\,\mathrm{d}u
  &= \ln S(t) - \ln S(t_{i-1}) = \ln S(t)
\end{aligned}$$

取指數，得到本章第一條重要結果與對應的密度：

$$\begin{aligned}
S(t) &= \exp\!\left[-\int_{t_{i-1}}^{t}\lambda^*(u)\,\mathrm{d}u\right], \\
f(t) &= \lambda^*(t)\,\exp\!\left[-\int_{t_{i-1}}^{t}\lambda^*(u)\,\mathrm{d}u\right]
\end{aligned}$$

三個立刻可用的推論。**其一，預報機率的公式**：「未來 $\Delta t$ 內至少發生
一次」的機率是 $P = 1 - \exp\bigl[-\int_t^{t+\Delta t}\lambda^*\bigr]$。第 9
章那些「未來七天內 $M\ge5$ 的機率 68%」全部是這條式子算出來的；它永遠落在
$[0,1)$，不管 $\lambda^*$ 多大。**其二，Poisson 是特例**：$\lambda^*\equiv\mu$
時積分變成 $\mu(t-t_{i-1})$，間隔服從指數分布、無記憶。**其三**，知道了每
一步的條件密度，連乘起來就是整份目錄的機率——那正是 10.2 節要做的事。

先用模擬把推導驗證一次。取一個簡單的自我激發強度：上一個事件剛發生在
$t=0$，之後 $\lambda^*(t)=\mu+\kappa\,g(t)$，$g$ 是正規化的 Omori 核。理論
存活曲線可以手算；模擬則完全不用這條公式，直接解 $\Lambda(t)=E$（$E$ 從
Exp(1) 抽），看兩者合不合：

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from gdms_toolkit.viz import ACCENT, PALETTE, QUAKE_COLOR, apply_layout

MU_D, KAPPA_D = 0.20, 3.0          # 背景率（次/天）、上一個事件的產能
C_D, P_D = 0.30, 1.40              # Omori 核參數（天）

def lam_demo(t):
    """單一觸發源的條件強度。"""
    return MU_D + KAPPA_D * (P_D - 1) / C_D * (1 + t / C_D) ** (-P_D)

def Lam_demo(t):
    """對應的 compensator（解析積分）。"""
    return MU_D * t + KAPPA_D * (1 - (1 + t / C_D) ** (-(P_D - 1)))

rng = np.random.default_rng(20)
n_sim = 40000
E = rng.exponential(1.0, n_sim)                 # 單位速率 Poisson 的第一個間隔
t_grid = np.linspace(0, 300, 300001)            # 用單調的 Λ 做數值反函數
t_sim = np.interp(E, Lam_demo(t_grid), t_grid)  # 解 Λ(t) = E

t_plot = np.logspace(-3, np.log10(60), 400)
S_theory = np.exp(-Lam_demo(t_plot))
S_emp = np.array([(t_sim > t).mean() for t in t_plot])
S_naive = np.exp(-lam_demo(0.0) * t_plot)       # 誤把 t=0 的率當常數
gap = np.abs(S_emp - S_theory).max()

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_plot, y=S_theory, mode="lines",
                         name="理論 exp(−∫λ*)",
                         line=dict(color=ACCENT, width=3)))
fig.add_trace(go.Scatter(x=t_plot, y=S_emp, mode="markers",
                         name=f"模擬經驗分布（{n_sim} 次）",
                         marker=dict(color=QUAKE_COLOR, size=5, opacity=0.7)))
fig.add_trace(go.Scatter(x=t_plot, y=S_naive, mode="lines",
                         name="若誤當成常數率 λ*(0)",
                         line=dict(color=PALETTE[3], width=2, dash="dot")))
apply_layout(fig, title=f"存活函數的驗證：最大偏差 {gap:.4f}",
             xaxis_title="距上一事件的時間（天）",
             yaxis_title="P（下一事件還沒來）",
             xaxis_type="log", yaxis_type="log",
             yaxis_range=[-2.2, 0.05], height=430, hovermode="x")
fig

藍線與紅點幾乎完全重合，推導沒問題。真正有教育意義的是那條點虛線：如果
偷懶把當下的強度 $\lambda^*(0)$ 當成常數往前推，會嚴重低估「下一個地震還
沒來」的機率，因為強度其實一直在衰減。**餘震序列的短期預報最容易踩這個
坑**——用剛發生時的高強度外推一整週，機率會高得離譜。積分不能用乘法取代。
順帶一提，`np.interp(E, Lam_demo(t_grid), t_grid)` 這一行其實是 10.5 節
「隨機時間變換」定理反過來跑：在變換後的時間軸上抽一個單位速率 Poisson
的間隔，再變換回真實時間。

## 10.2 概似函數的來源

有了條件密度，就可以問統計學最基本的問題：**給定一組參數，觀測到這份目錄
的機率有多大**？答案就是概似函數，而它的形式漂亮到讓人懷疑是不是作弊。

設觀測窗為 $[0,T]$，觀測到 $N$ 個事件 $t_1<\cdots<t_N\le T$。整份資料的
聯合密度用條件機率的鏈式法則拆開（記 $t_0\equiv0$）：

$$L \;=\; \left[\prod_{i=1}^{N} f(t_i \mid H_{t_i})\right]
  \times P\bigl(N(T)-N(t_N)=0\bigr)$$

最後那一項最容易被忘記，卻非常重要：**你不只觀測到 N 個事件發生了，你還
觀測到「其他時候沒有發生」**。這個「沒有發生」也是資訊，也要進概似；忘掉
它，模型就沒有理由不把強度調到無窮大。逐項代入 10.1 節的結果：

$$\begin{aligned}
f(t_i \mid H_{t_i}) &= \lambda^*(t_i)\,
  \exp\!\left[-\int_{t_{i-1}}^{t_i} \lambda^*(u)\,\mathrm{d}u\right], \\
P\bigl(N(T)-N(t_N)=0\bigr) &=
  \exp\!\left[-\int_{t_N}^{T} \lambda^*(u)\,\mathrm{d}u\right]
\end{aligned}$$

連乘之後，指數項裡的積分**首尾相接**——這是整段推導的樞紐：

$$\begin{aligned}
L &= \left[\prod_{i=1}^{N} \lambda^*(t_i)\right]
  \exp\!\left[-\sum_{i=1}^{N}\int_{t_{i-1}}^{t_i}\lambda^*
    -\int_{t_N}^{T}\lambda^*\right] \\
  &= \left[\prod_{i=1}^{N} \lambda^*(t_i)\right]
  \exp\!\left[-\int_{0}^{T} \lambda^*(u)\,\mathrm{d}u\right]
\end{aligned}$$

那一串區間 $(0,t_1],(t_1,t_2],\ldots,(t_N,T]$ 剛好把 $[0,T]$ 鋪滿，所以合併
成一個積分。取對數：

$$\ln L \;=\; \sum_{i=1}^{N} \ln \lambda^*(t_i)
  \;-\; \int_{0}^{T} \lambda^*(t)\,\mathrm{d}t$$ (eq:pp-loglik)

兩項，沒有第三項。這條式子是第二部的心臟，值得逐字讀：**第一項是獎勵**
——事件真的發生的那些時刻，模型說的強度愈高，$\ln L$ 愈大，它鼓勵模型
「說對地方」；**第二項是懲罰**——$\int_0^T\lambda^*$ 是模型預期的總事件數，
模型到處喊高，這一項就大。兩項合起來是一場**押注的結算**：你有固定的籌碼
（總預期事件數），要把它押在時間軸（以及後面的空間軸、規模軸）上，押中
有賞、亂押有罰。整個預報檢驗學（第 18 章）的精神全藏在這兩項的張力裡。

### 推廣到時空標記版

真實目錄不只有時間。此時條件強度變成四維的密度 $\lambda^*(t,x,y,m)$，單位
是「次／(天·平方度·規模單位)」。推導完全平行，只要把「$(t,t+\mathrm{d}t)$
內有沒有事件」換成「這個四維小盒裡有沒有事件」，結果是

$$\ln L \;=\; \sum_{i=1}^{N} \ln \lambda^*(t_i, x_i, y_i, m_i)
  \;-\; \int_{0}^{T}\!\!\int_{S}\!\int_{m_0}^{\infty}
  \lambda^*\,\mathrm{d}m\,\mathrm{d}x\,\mathrm{d}y\,\mathrm{d}t$$

逐步展開放在 10.11 節附錄。這裡只點出三件實務上會咬人的事。**其一，$S$ 是
誰很要緊**：積分區域是「你宣稱在預報的區域」，不是「有事件的區域」；把 $S$
縮到只包含事件密集區，$\ln L$ 會虛假地變好看。**其二，邊界效應**：研究區外
或研究期前的事件仍會觸發區內的事件，所以要進 $H_t$，卻不該進第一項的求和
——這種事件叫 complementary event，是實作 ETAS 的標準處理（Jalilian 2019）。
**其三，$\ln L$ 的絕對值沒有意義**，只有同一份目錄、同一個 $m_0$、同一個
$S$ 與 $T$ 之下的差值才能比；除以 $N$ 得到的「每事件資訊增益」才是可以跨
研究溝通的量（第 18 章擁有這個定義）。

最後補一句歷史。用點過程 MLE 估參數，是 Ogata 在 1983 年為修正 Omori 律
引進的。在那之前大家的做法是「先把餘震分 bin 數個數、再對 log–log 做迴歸」，
而分 bin 有兩個致命傷：bin 寬度是任意的，而且早期高活動期被壓成一個點。
{eq}`eq:pp-loglik` 直接吃事件時間本身，不需要任何 bin——這是地震統計方法論
上一次真正的升級。

## 10.3 三種記憶結構

{eq}`eq:pp-loglik` 對**任何** $\lambda^*$ 都成立。所以造模型這件事，就化約
成一個問題：**$\lambda^*$ 怎麼依賴歷史**？文獻上的答案可以歸成三大類，
而三類的差別只有一個字——記憶的**方向**。

（一）**無記憶：Poisson 過程**，$\lambda^*(t)=\mu$。歷史完全不影響未來，
間隔服從指數分布。它顯然是錯的（餘震的存在就是反證），但它是所有預報模型
必須先打敗的**虛無假設**，也是 PSHA（第 21 章）長期危害計算的預設骨架。
空間非均勻、時間恆定的版本 $\lambda^*(x,y)=\mu(x,y)$ 叫非齊次 Poisson，
是 CSEP 時間獨立預報的標準形式。

（二）**正記憶：self-exciting**（Hawkes／ETAS）：

$$\lambda^*(t) \;=\; \mu \;+\;
  \sum_{i:\,t_i < t} \kappa(m_i)\, g(t - t_i)$$

每個過去事件都**推高**未來的強度，推高量隨時間衰減。Hawkes（1971）提出
這個結構時想的就是地震，但沒寫下規模相依的產能；Ogata（1988）把產能
$\kappa(m)=Ae^{\alpha(m-m_0)}$ 與 Omori 核裝上去，就成了 ETAS——第 13 章的
主角。餘震、二次餘震、前震全部從這一條式子自動長出來。

（三）**負記憶：self-correcting**（應力釋放）：

$$\lambda^*(t) \;=\; \exp\bigl\{a_s + b_s\,[\,t - c_s S(t)\,]\bigr\}$$

$S(t)$ 是到 $t$ 為止的累積應力釋放量（實務上取累積地震矩或 Benioff 應變）。
這是 Vere-Jones（1978）的**應力釋放模型**：應力以固定速率累積、每次地震
釋放一塊，強度隨累積應力上升、隨事件**下降**。它是彈性回跳理論最直接的
點過程翻譯，也是第 20 章複發模型的統計版本。（文獻原本用 $a,b,c$，與 GR 的
$b$ 值、Omori 的 $c$ 撞名，本書一律改寫成 $a_s,b_s,c_s$。）

### 同一個框架的反號

把（二）與（三）並排，會發現它們其實是同一件事的兩個方向。兩者都可以寫成

$$\lambda^*(t) \;=\; \Phi\!\left[\,\eta_0 \;\pm\;
  \sum_{i:\,t_i<t} w(t-t_i,\,m_i)\,\right]$$

其中 $\eta_0$ 是基線、$w$ 是每個過去事件的貢獻。ETAS 取加號、$\Phi$ 是恆等
函數；應力釋放取減號、$\Phi$ 是指數函數。**地震觸發地震**與**地震消耗應力**，
在點過程的語言裡是同一條句型的正反兩讀。

這也解釋了它們各自的適用尺度：短期（天～週）看到的是觸發，因為應力重分布
的正效應在餘震區內壓倒一切；長期（數十～數百年）看到的是耗竭，因為一次大
破裂真的把斷層上的應力放掉了。兩者不矛盾，只是尺度不同；真正的難題是中間
那段（月～年），那正是第 15、16 章那些中期模型要面對的灰色地帶。還有第四類
值得一提：**更新過程**（renewal process），$\lambda^*$ 只依賴「距上一個事件
多久」，不管更早的歷史——BPT、Weibull、對數常態複發模型都屬於這一類
（第 20 章）。它是一種有限記憶：記得上一次，忘記上上次。

下面用**同一份**合成事件序列，把三種強度並排畫出來。事件位置完全一樣，
差別只在模型怎麼解讀它們：

In [ ]:
rng = np.random.default_rng(31)
T_DEMO, M0, BETA = 60.0, 3.0, np.log(10.0)           # 門檻規模、GR 斜率（b = 1）

t_ev = np.sort(rng.uniform(0, T_DEMO, 16))
m_ev = M0 + rng.exponential(1 / BETA, t_ev.size)
grid = np.linspace(0, T_DEMO, 3000)

MU_B = 0.30                                          # 共同的基線率
lam_poi = np.full_like(grid, MU_B)

A_B, ALPHA_B, C_B, P_B = 0.45, 0.80, 0.30, 1.40      # self-exciting
lam_se = np.full_like(grid, MU_B)
for ti, mi in zip(t_ev, m_ev):
    msk = grid > ti
    lam_se[msk] += (A_B * np.exp(ALPHA_B * (mi - M0)) * (P_B - 1) / C_B
                    * (1 + (grid[msk] - ti) / C_B) ** (-P_B))

B_S = 0.30                                           # self-correcting
C_S = T_DEMO / t_ev.size                             # 加載與釋放長期平衡
n_past = np.searchsorted(t_ev, grid, side="right")
lam_sc = np.exp(np.log(MU_B) + B_S * (grid - C_S * n_past))

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    subplot_titles=("無記憶：Poisson λ* = μ",
                                    "正記憶：self-exciting（Hawkes／ETAS）",
                                    "負記憶：self-correcting（應力釋放）"))
for row, (lam, color) in enumerate(
        [(lam_poi, PALETTE[0]), (lam_se, PALETTE[1]), (lam_sc, PALETTE[2])], 1):
    fig.add_trace(go.Scatter(x=grid, y=lam, mode="lines", showlegend=False,
                             line=dict(color=color, width=1.8)), row=row, col=1)
    fig.add_trace(go.Scatter(x=t_ev, y=np.zeros_like(t_ev), mode="markers",
                             name="事件", showlegend=(row == 1),
                             marker=dict(color=QUAKE_COLOR, size=7,
                                         symbol="triangle-up")), row=row, col=1)
    fig.update_yaxes(title_text="λ*（次/天）", row=row, col=1)
fig.update_xaxes(title_text="時間（天）", row=3, col=1)
apply_layout(fig, title=f"同一份事件序列（{t_ev.size} 個），三種記憶結構",
             height=640, hovermode="x")
fig

三張圖的紅色三角形位置完全相同。上圖是一條水平線——事件的到來不改變任何
事。中圖每來一個事件就往上跳，跳多高看規模、跳完按 Omori 律衰減。下圖相反：
事件一來就往下掉，之後隨著應力重新累積慢慢爬升。值得注意的是中圖與下圖的
**時間尺度感**完全不同：self-exciting 的峰值集中在事件剛發生後的極短時間內
（Omori 核在 $t\to0$ 附近極陡）；self-correcting 則是緩慢的鋸齒，週期由加載
速率決定。你覺得一份真實目錄像哪一張，取決於你把時間軸拉多長。

## 10.4 標記與可分離性

現在把規模這個**標記**（mark）加進來。最常用、幾乎所有作業化模型採用的
假設是**可分離性**（separability）：

$$\lambda^*(t,x,y,m) \;=\; \lambda^*(t,x,y)\, s(m),
  \qquad s(m) = \beta\,e^{-\beta(m - m_0)}$$

白話講：**什麼時候在哪裡發生地震，跟它會多大，是兩件獨立的事**。地震的
規模永遠從同一個 GR 分布抽出來，不管它是背景事件還是餘震、不管當下強度
多高。$s(m)$ 是 Gutenberg–Richter 律的機率密度版本，
$\int_{m_0}^{\infty}s(m)\,\mathrm{d}m=1$，而 $\beta = b\ln 10$。第 11 章會
完整處理 GR 律與 $b$ 值估計，這裡只需要它是個歸一化的密度。把可分離的
形式代進時空標記版的 $\ln L$：

$$\begin{aligned}
\ln L &= \sum_{i=1}^{N}\ln\bigl[\lambda^*(t_i,x_i,y_i)\,s(m_i)\bigr]
  - \int_0^T\!\!\int_S\!\int_{m_0}^{\infty}
    \lambda^*(t,x,y)\,s(m)\,\mathrm{d}m\,\mathrm{d}x\,\mathrm{d}y\,\mathrm{d}t \\
  &= \sum_{i=1}^{N}\ln\lambda^*(t_i,x_i,y_i) + \sum_{i=1}^{N}\ln s(m_i)
    - \int_0^T\!\!\int_S \lambda^*(t,x,y)
      \underbrace{\left[\int_{m_0}^{\infty} s(m)\,\mathrm{d}m\right]}_{=\,1}
      \mathrm{d}x\,\mathrm{d}y\,\mathrm{d}t
\end{aligned}$$

中間那個內層積分等於 1，於是規模從積分項裡**整個消失**，概似漂亮地裂成
兩塊 $\ln L = \ell_1(\beta) + \ell_2(\theta)$，前者只管規模、後者只管時空：

$$\begin{aligned}
\ell_1(\beta) &= \sum_{i=1}^{N}\ln s(m_i)
  = N\ln\beta - \beta\sum_{i=1}^{N}(m_i - m_0), \\
\ell_2(\theta) &= \sum_{i=1}^{N}\ln\lambda^*_\theta(t_i,x_i,y_i)
  - \int_0^T\!\!\int_S \lambda^*_\theta\,\mathrm{d}x\,\mathrm{d}y\,\mathrm{d}t
\end{aligned}$$

$\ell_1$ 只含 $\beta$，$\ell_2$ 只含時空參數 $\theta$，**兩者可以完全獨立地
最佳化**。這在實務上是件大事：ETAS 有八個時空參數要用擬牛頓法慢慢爬，
而 $\beta$ 完全不必參與那場苦戰。

### $\beta$ 的封閉解

更好的是，$\ell_1$ 可以手算到底。對 $\beta$ 微分並令其為零：

$$\frac{\partial \ell_1}{\partial \beta}
  \;=\; \frac{N}{\beta} - \sum_{i=1}^{N}(m_i - m_0) \;=\; 0
  \quad\Longrightarrow\quad
  \hat\beta \;=\; \frac{N}{\sum_{i=1}^{N}(m_i - m_0)}$$

二階導數 $-N/\beta^2 < 0$，確定是極大值。換算回 $b$ 值：

$$\hat b \;=\; \frac{\hat\beta}{\ln 10}
  \;=\; \frac{1}{\ln 10 \cdot \overline{(m - m_0)}}$$

這就是有名的 **Aki（1965）估計式**：$b$ 值只依賴「平均規模超出門檻多少」
這一個數字。它從最大概似法自然掉出來，不是湊出來的經驗公式。第 11 章會
深入它的偏差修正、分箱版本、不確定度與 $M_c$ 的交互作用——那裡才是主場。

### 可分離性是假設，不是定理

這一節從頭到尾建立在「規模與時空獨立」上，而這個假設**很可能是錯的**，
只是錯得還不夠明顯：有研究指出大震前 $b$ 值會下降（也就是 $s$ 依賴於
歷史）；餘震與背景活動的 $b$ 值是否相同至今有爭議，而其中一部分爭議其實
是除叢演算法造成的假象（第 12 章）。如果規模真的與歷史有關，$\beta$ 就
不能分離出去，整套估計流程要重寫。目前作業化模型幾乎一律採用可分離性，
理由是它有效、穩定，而且放棄它之後的模型至今沒有證明自己更好——這是實用
主義，不是真理。

## 10.5 compensator 與殘差分析

模型寫好、參數估好之後，下一個問題是：**它對嗎**？點過程有一個非常優雅的
答案，優雅到幾乎像魔術。定義 **compensator**（補償子，或稱累積強度）：

$$\Lambda(t) \;=\; \int_0^t \lambda^*(u)\,\mathrm{d}u$$

它是模型預期的累積事件數。名字的由來是 $N(t)-\Lambda(t)$ 是一個平均為零的
鞅（martingale）——$\Lambda$ 恰好「補償」掉 $N$ 的趨勢，而這個性質是下面
所有殘差工具的來源。

### 隨機時間變換定理

> **定理**（Meyer 1971；Papangelou 1972）。設 $\lambda^*(t)>0$ 且
> $\Lambda(\infty)=\infty$，定義 $\tau_i=\Lambda(t_i)$。則
> $\{\tau_1,\tau_2,\ldots\}$ 是一個**單位速率的 Poisson 過程**。

這句話的力量在於：**不管你的模型多複雜、多非線性、記憶多長，只要它是對的，
把時間軸按 $\Lambda$ 拉伸之後，事件就變成最簡單的隨機點**。所有模型的檢驗
因此被化約成同一個問題：這串點看起來像不像單位速率 Poisson？

**證明骨架**。因 $\lambda^*>0$，$\Lambda$ 嚴格遞增且連續，反函數存在。
看變換後的第 $i$ 個間隔：

$$\begin{aligned}
P(\tau_{i+1} - \tau_i > s \mid H_{t_i})
  &= P\bigl(t_{i+1} > \Lambda^{-1}(\tau_i + s) \bigm| H_{t_i}\bigr) \\
  &= \exp\!\left[-\int_{t_i}^{\Lambda^{-1}(\tau_i+s)}
     \lambda^*(u)\,\mathrm{d}u\right] \\
  &= \exp\bigl[-\bigl(\Lambda(\Lambda^{-1}(\tau_i+s)) - \Lambda(t_i)\bigr)\bigr]
  \;=\; e^{-s}
\end{aligned}$$

第二個等號用的正是 10.1 節推出的存活函數，第三個等號只是 compensator 的
定義。結論：變換後的間隔是 i.i.d. 的 Exp(1)，而「間隔 i.i.d. Exp(1)」就是
單位速率 Poisson 過程的定義。完整版（包括為什麼可以用同一個論證處理全部
間隔）放在 10.11 節附錄。由此立刻得到兩個可畫的診斷量：令
$\Delta\tau_j=\tau_j-\tau_{j-1}$，則

$$U_j \;=\; 1 - e^{-\Delta\tau_j} \;\sim\; U(0,1),
  \quad \text{i.i.d.}$$ (eq:time-rescale)

這一步是**機率積分變換**：Exp(1) 的累積分布函數就是 $1-e^{-s}$，把隨機
變數餵給自己的 cdf 就得到均勻分布。

### 三種診斷圖

**一、$\tau_j$ 對 $j$ 的圖**。模型正確時點應落在 $y=x$ 附近。**低於直線的
區段代表「相對於模型的寧靜」（quiescence）——實際事件比模型預期的少；高於
直線則是相對活化**。「相對」兩個字很要緊：它不是絕對的活動高低，而是扣掉
模型（例如餘震衰減）之後剩下的部分。Ogata & Zhuang（2006）指出，先有一個
能描述「正常叢集」的參考模型，才談得上偵測異常寧靜，否則所謂寧靜可能只是
餘震正常衰減的錯覺。

**二、$U_j$ 的 Q–Q 圖**（對 $U(0,1)$）。落在對角線上就過關，也可以直接跑
Kolmogorov–Smirnov 檢定得到一個 $p$ 值。**三、$U_j$ 的自相關**。時間變換
只保證邊際分布均勻，正確的模型還要求**獨立**；畫 $U_j$ 對 $U_{j-1}$ 的
散布圖若出現結構，代表殘餘的相依性沒被模型吃掉。

### 殘差的一般形式

上面是一維時間版本。時空版本用加權殘差寫成統一形式：對任一權重函數 $h$
與任一時空區塊 $I\times B$，

$$R(I\times B;\,h) \;=\;
  \sum_{i:\,t_i \in I,\,(x_i,y_i)\in B} h(t_i,x_i,y_i)
  \;-\; \iiint_{I\times B} h\,\lambda^*\,
  \mathrm{d}x\,\mathrm{d}y\,\mathrm{d}t$$

鞅性質保證 $E[R]=0$（推導見附錄）。三個常用選擇：

| 名稱 | 權重 $h$ | 特性 |
|---|---|---|
| raw residual | $1$ | 「觀測數 − 期望數」，最直觀，但被高強度區主宰 |
| reciprocal residual | $1/\lambda^*$ | 反過來，對背景區的一顆事件極度敏感 |
| Pearson residual | $1/\sqrt{\lambda^*}$ | 兩者的折衷，變異數穩定，做圖首選 |

下面把診斷圖畫出來。左欄餵**正確**的模型，右欄餵一個故意錯設的模型（把
叢集完全忽略、當成均勻 Poisson，但總期望數校準得一模一樣）。目錄是用 10.6
節要介紹的分支法模擬出來的——先看診斷怎麼讀，演算法下一節再講：

In [ ]:
C_OM, P_OM = 0.30, 1.40                # 本章共用的 Omori 核參數（天）

def simulate_branching(mu, A, alpha, T, rng, c=C_OM, p=P_OM):
    """分支（cluster）法模擬時間型 ETAS，回傳 (時間, 規模)。"""
    t = list(rng.uniform(0, T, rng.poisson(mu * T)))       # 第 0 代：背景
    m = list(M0 + rng.exponential(1 / BETA, len(t)))
    todo = list(range(len(t)))
    while todo:
        i = todo.pop()
        for _ in range(rng.poisson(A * np.exp(alpha * (m[i] - M0)))):
            dt = c * ((1 - rng.random()) ** (-1 / (p - 1)) - 1)   # 反函數法
            if t[i] + dt < T:
                t.append(t[i] + dt)
                m.append(M0 + rng.exponential(1 / BETA))
                todo.append(len(t) - 1)
    order = np.argsort(t)
    return np.asarray(t)[order], np.asarray(m)[order]

def compensator(t, ts, ms, mu, A, alpha, c=C_OM, p=P_OM):
    """Λ(t) = ∫λ*：Omori 核的積分有解析形式。"""
    past = ts < t
    trig = (A * np.exp(alpha * (ms[past] - M0))
            * (1 - (1 + (t - ts[past]) / c) ** (-(p - 1))))
    return mu * t + trig.sum()

MU_R, ALPHA_R, T_R, N_RATIO = 0.30, 0.80, 400.0, 0.65
A_R = N_RATIO * (BETA - ALPHA_R) / BETA          # 由分支比反推產能尺度

t_cat, m_cat = simulate_branching(MU_R, A_R, ALPHA_R, T_R,
                                  np.random.default_rng(42))
N_cat = t_cat.size
j_idx = np.arange(1, N_cat + 1)
q_theory = (j_idx - 0.5) / N_cat

tau_ok = np.array([compensator(ti, t_cat, m_cat, MU_R, A_R, ALPHA_R)
                   for ti in t_cat])
tau_bad = N_cat * t_cat / T_R                    # 錯設：均勻 Poisson，總數校準

fig = make_subplots(rows=2, cols=2, vertical_spacing=0.13, horizontal_spacing=0.10,
                    subplot_titles=("正確模型：τ 對 j", "錯設模型：τ 對 j",
                                    "正確模型：U 的 Q–Q 圖",
                                    "錯設模型：U 的 Q–Q 圖"))
ks = {}
for col, (tau, name, color) in enumerate(
        [(tau_ok, "正確", PALETTE[2]), (tau_bad, "錯設", PALETTE[1])], 1):
    fig.add_trace(go.Scatter(x=j_idx, y=tau, mode="lines", showlegend=False,
                             line=dict(color=color, width=1.6)), row=1, col=col)
    fig.add_trace(go.Scatter(x=j_idx, y=j_idx, mode="lines", showlegend=False,
                             line=dict(color="#999999", width=1, dash="dash")),
                  row=1, col=col)
    u = np.sort(1 - np.exp(-np.diff(np.concatenate([[0.0], tau]))))
    ks[name] = np.abs(u - q_theory).max()
    fig.add_trace(go.Scattergl(x=q_theory, y=u, mode="markers", showlegend=False,
                               marker=dict(color=color, size=4, opacity=0.7)),
                  row=2, col=col)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", showlegend=False,
                             line=dict(color="#999999", width=1, dash="dash")),
                  row=2, col=col)
    fig.update_xaxes(title_text="事件序號 j", row=1, col=col)
    fig.update_xaxes(title_text="理論分位數", row=2, col=col)
fig.update_yaxes(title_text="τ = Λ(t)", row=1, col=1)
fig.update_yaxes(title_text="U 的樣本分位數", row=2, col=1)
apply_layout(fig, height=680, hovermode="closest",
             title=f"隨機時間變換診斷（N = {N_cat}）："
                   f"KS 統計量 正確 {ks['正確']:.3f}／錯設 {ks['錯設']:.3f}")
fig

左欄是正確模型：$\tau$ 對 $j$ 貼著虛線走，Q–Q 圖也貼著對角線，KS 統計量
很小。右欄是錯設模型：$\tau$–$j$ 曲線在叢集發生時**陡升**（短時間衝出一堆
事件，均勻模型的 $\tau$ 卻只按時間線性成長，於是相對直線先落後又追上），
Q–Q 圖則在低分位數嚴重偏離——因為叢集造成大量極短的間隔，$U_j$ 堆在接近 0
的地方。這正是「模型沒有描述叢集」的簽名。

兩個必須馬上補的警語。**第一，殘差圖裡的偏離不必然是物理訊號**：目錄不
完整（大震後幾小時測不到小地震）、規模尺度換算、測網增設或撤站，都會在
$\tau$–$j$ 圖上製造出漂亮的「寧靜」。**第二，上圖的參數是已知的真值**；
實務上參數是從同一份資料估出來的，殘差因此帶有樂觀偏誤——模型已經為了配合
這份目錄調過參數了。嚴格的檢驗要用**前瞻**（prospective）資料，這是第 18
章與 CSEP 制度的核心精神。

## 10.6 模擬：兩條對偶的路

模型會生資料，才算真的活著。模擬在地震預報裡有三個不可取代的用途：產生
預報的不確定度區間（第 17 章的 N 檢驗、S 檢驗全靠它）、檢查估計程序有沒有
偏差、以及回答「如果世界真的長這樣，我們會看到什麼」。點過程有兩條主要的
模擬路線，想法完全不同，卻收斂到同一個分布。

### 路線一：thinning（稀疏化，Lewis & Shedler 1979；Ogata 1981）

想法是「先多生一點，再丟掉一些」。假設在 $[t,t']$ 上找得到一個上界
$\bar\lambda \ge \lambda^*(s)$（對所有 $s\in[t,t']$），那麼：

1. 用速率 $\bar\lambda$ 的**齊次** Poisson 過程產生提案點（間隔從
   $\mathrm{Exp}(1/\bar\lambda)$ 抽）；
2. 對每個提案點 $s$，抽 $V \sim U(0,\bar\lambda)$；
3. 若 $V \le \lambda^*(s)$ 就**接受**，否則丟棄；
4. 每接受一個事件、或每丟棄一個提案點，就重新計算上界，繼續。

對 self-exciting 模型，上界的選法特別簡單：因為觸發核在事件之間單調遞減，
**當下的強度值 $\lambda^*(t^+)$ 本身就是未來的上界**（只要還沒有新事件被
接受）。

**為什麼這樣是對的**？最乾淨的論證是幾何的。把提案點畫在 $(s,v)$ 平面上，
它們構成一個在 $[t,t']\times[0,\bar\lambda]$ 上**密度均勻**的二維 Poisson
過程（一維均勻 × 高度均勻）。二維 Poisson 過程限制在任一子區域上仍是
Poisson 過程；取「曲線 $\lambda^*(s)$ 以下」這個子區域，它在 $s$ 方向的
邊際強度就是曲線的高度 $\lambda^*(s)$。所以被接受的點恰好構成強度為
$\lambda^*$ 的過程。又因為接受與否只依賴 $s$ 之前的歷史，遞迴做下去，
整條軌跡的條件強度都正確。下面把這張「幾何論證」直接畫出來——這張圖本身
就是證明：

In [ ]:
def lam_star_t(s, ts, ms, mu, A, alpha, c=C_OM, p=P_OM):
    """時間型 ETAS 的條件強度（ts 只含 s 之前的事件）。"""
    if ts.size == 0:
        return mu
    return mu + np.sum(A * np.exp(alpha * (ms - M0)) * (p - 1) / c
                       * (1 + (s - ts) / c) ** (-p))

def simulate_thinning(mu, A, alpha, T, rng, record=False):
    """Ogata thinning：回傳 (時間, 規模[, 提案紀錄])。"""
    t, ts, ms, rec = 0.0, [], [], []
    while True:
        at, am = np.asarray(ts), np.asarray(ms)
        lam_bar = lam_star_t(t, at, am, mu, A, alpha)      # 上界＝當下強度
        s = t + rng.exponential(1.0 / lam_bar)
        if s > T:
            break
        v = rng.uniform(0.0, lam_bar)
        accept = v <= lam_star_t(s, at, am, mu, A, alpha)
        if record:
            rec.append((t, s, v, lam_bar, accept))
        if accept:
            ts.append(s)
            ms.append(M0 + rng.exponential(1 / BETA))
        t = s
    out = (np.asarray(ts), np.asarray(ms))
    return out + (rec,) if record else out

t_th, m_th, rec = simulate_thinning(0.30, A_R, ALPHA_R, 25.0,
                                    np.random.default_rng(5), record=True)
gg = np.linspace(0, 25, 4000)
lam_g = np.array([lam_star_t(s, t_th[t_th < s], m_th[t_th < s],
                             0.30, A_R, ALPHA_R) for s in gg])

fig = go.Figure()
step_x, step_y = [], []
for t0, s, v, lb, acc in rec:
    step_x += [t0, s, None]
    step_y += [lb, lb, None]
fig.add_trace(go.Scatter(x=step_x, y=step_y, mode="lines", name="上界 λ̄（階梯）",
                         line=dict(color="#999999", width=1.4, dash="dot")))
fig.add_trace(go.Scatter(x=gg, y=lam_g, mode="lines", name="條件強度 λ*(t)",
                         line=dict(color=ACCENT, width=2)))
acc_pts = [(s, v) for _, s, v, _, a in rec if a]
rej_pts = [(s, v) for _, s, v, _, a in rec if not a]
for pts, name, color, sym in [(rej_pts, "提案被拒絕", PALETTE[3], "x"),
                              (acc_pts, "提案被接受", QUAKE_COLOR, "circle")]:
    px, py = zip(*pts)
    fig.add_trace(go.Scatter(x=list(px), y=list(py), mode="markers", name=name,
                             marker=dict(color=color, size=8, symbol=sym,
                                         opacity=0.85)))
apply_layout(fig, yaxis_type="log",
             title=f"Thinning：{len(rec)} 個提案點，接受 {len(acc_pts)} 個"
                   f"（接受率 {len(acc_pts) / len(rec):.0%}）",
             xaxis_title="時間（天）", yaxis_title="強度（次/天，對數軸）",
             height=460, hovermode="closest")
fig

灰色階梯是每一段的上界 $\bar\lambda$，藍線是真正的條件強度 $\lambda^*$，
每個提案點的高度 $v$ 是均勻抽的：**落在藍線以下的被接受（紅圓），以上的被
丟棄（叉）**。上界階梯只在事件被接受時往上跳，之後隨著每次拒絕逐步下修
——這就是「用當下強度當上界」的實際樣子；叢集愈強、峰值愈尖，浪費的提案點
就愈多。thinning 有兩個優點值得記住：它**不需要 $\Lambda$ 的解析形式**，
也不需要任何反函數，只要求你會算 $\lambda^*(s)$ 與一個上界。這讓它幾乎可以
套用在任何模型上，包括那些積分算不出來的。

### 路線二：branching（分支／叢集法）

第二條路完全不碰強度函數，而是照著模型的**故事**走：

1. **第 0 代（移民）**：在 $[0,T]$ 上依背景率抽一個 Poisson 過程得到背景
   事件，規模從 $s(m)$ 獨立抽。
2. **繁殖**：對上一代每個事件 $i$，抽後代數
   $N_i\sim\mathrm{Poisson}\bigl(\kappa(m_i)\bigr)$；每個後代的時間延遲從
   $g$ 抽、空間位移從空間核抽、規模從 $s(m)$ 抽（**與親代規模無關**）。
3. **遞迴**：把新的一代當成上一代，重複，直到某代沒有新事件；最後丟掉
   落在時空窗外的事件。

只要分支比小於 1，遞迴必然終止（第 13 章會推導分支比並說明臨界性）。這條
路線的教學價值極高：寫完這三十行程式，你會**親眼看到二次餘震自己長出來**，
而你從頭到尾沒有寫過任何一條「二次餘震」的規則。兩條路的分工也很清楚
——thinning 對應**數學**（條件強度的定義），branching 對應**物理故事**
（誰觸發了誰）。branching 還免費附送每個事件的親代標籤；thinning 則適用
面更廣。

### 完整推導：反函數法抽樣

branching 法需要「從 Omori 核抽一個時間延遲」，這靠的是**反函數法**
（inverse transform sampling）。先證明它為什麼對：設 $X$ 的累積分布函數
$G$ 連續且嚴格遞增、$U\sim U(0,1)$，則 $G^{-1}(U)$ 與 $X$ 同分布。證明只有
一行——$P\bigl(G^{-1}(U)\le x\bigr) = P\bigl(U\le G(x)\bigr) = G(x)$，
最後一步用的是均勻分布的 cdf 就是恆等函數。

**套用一：Omori 時間核**。密度 $g(t)=\frac{p-1}{c}(1+\frac{t}{c})^{-p}$，
累積分布函數

$$\begin{aligned}
G(t) &= \int_0^t \frac{p-1}{c}\left(1+\frac{u}{c}\right)^{-p}\mathrm{d}u
      = \left[-\left(1+\frac{u}{c}\right)^{-(p-1)}\right]_0^t
      = 1 - \left(1+\frac{t}{c}\right)^{-(p-1)}
\end{aligned}$$

令 $U = G(\Delta t)$ 反解：

$$\begin{aligned}
\left(1+\frac{\Delta t}{c}\right)^{-(p-1)} &= 1 - U \\
1 + \frac{\Delta t}{c} &= (1-U)^{-1/(p-1)} \\
\Delta t &= c\left[(1-U)^{-1/(p-1)} - 1\right]
\end{aligned}$$

這條式子在上面的 `simulate_branching` 裡就是那一行
`dt = c * ((1 - rng.random()) ** (-1 / (p - 1)) - 1)`。注意它清楚顯示
$p>1$ 是必要的：$p\le1$ 時指數 $-1/(p-1)$ 的符號翻轉，$G$ 根本不收斂到 1，
Omori 核不可正規化——「擬出 $p<1$」在點過程框架裡不是慢衰減，是模型設定
壞掉（10.9 節、第 12 章）。**套用二：冪次空間核的半徑**。空間密度

$$f(x,y) = \frac{q-1}{\pi\sigma}\left(1+\frac{r^2}{\sigma}\right)^{-q},
  \qquad \sigma = D\,e^{\gamma(m-m_0)},\quad r^2 = x^2+y^2$$

核是各向同性的，所以角度從 $U(0,2\pi)$ 均勻抽，只剩半徑要處理。半徑的
累積分布函數要記得帶 Jacobian $2\pi r$：

$$\begin{aligned}
P(R \le r) &= \int_0^{r} 2\pi u \,\frac{q-1}{\pi\sigma}
  \left(1+\frac{u^2}{\sigma}\right)^{-q}\mathrm{d}u \\
  &= \int_0^{r^2/\sigma} (q-1)(1+w)^{-q}\,\mathrm{d}w
     \qquad (w = u^2/\sigma) \\
  &= 1 - \left(1+\frac{r^2}{\sigma}\right)^{-(q-1)}
\end{aligned}$$

形式與 Omori 核一模一樣（把 $t/c$ 換成 $r^2/\sigma$、$p$ 換成 $q$），
反解得 $r = \sqrt{\sigma\bigl[(1-U)^{-1/(q-1)}-1\bigr]}$。同樣地，$q>1$ 是
可正規化的必要條件。

### 兩條路真的等價嗎？

理論上，self-exciting 點過程與「Poisson 移民 + 分支後代」的叢集過程是同一
個東西（Hawkes & Oakes 1974 的表現定理）。但理論歸理論，動手驗證一次比較
踏實。下面用**完全相同的參數**跑兩套模擬各數百份目錄，比較統計量：

In [ ]:
MU_C, T_C, N_REP = 0.30, 60.0, 260
rng_b = np.random.default_rng(101)
rng_t = np.random.default_rng(202)

n_branch, n_thin, dt_branch, dt_thin = [], [], [], []
for _ in range(N_REP):
    tb, _mb = simulate_branching(MU_C, A_R, ALPHA_R, T_C, rng_b)
    tt, _mt = simulate_thinning(MU_C, A_R, ALPHA_R, T_C, rng_t)
    n_branch.append(tb.size)
    n_thin.append(tt.size)
    dt_branch.append(np.diff(tb))
    dt_thin.append(np.diff(tt))

n_branch, n_thin = np.array(n_branch), np.array(n_thin)
d_b = np.log10(np.concatenate(dt_branch) + 1e-6)
d_t = np.log10(np.concatenate(dt_thin) + 1e-6)
edges = np.arange(0, max(n_branch.max(), n_thin.max()) + 6, 5)
xs = np.linspace(-6, 2, 300)
ecdf_b = np.searchsorted(np.sort(d_b), xs) / d_b.size
ecdf_t = np.searchsorted(np.sort(d_t), xs) / d_t.size
d_max = np.abs(ecdf_b - ecdf_t).max()

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.11,
                    subplot_titles=("每份目錄的事件數 N", "事件間隔的經驗累積分布"))
for name, arr, ec, color in [("branching", n_branch, ecdf_b, PALETTE[0]),
                             ("thinning", n_thin, ecdf_t, PALETTE[1])]:
    h, _ = np.histogram(arr, bins=edges)
    fig.add_trace(go.Bar(x=(edges[:-1] + edges[1:]) / 2, y=h / arr.size,
                         name=name, legendgroup=name,
                         marker=dict(color=color, opacity=0.55)), row=1, col=1)
    fig.add_trace(go.Scatter(x=xs, y=ec, mode="lines", name=name,
                             legendgroup=name, showlegend=False,
                             line=dict(color=color, width=2)), row=1, col=2)
fig.update_xaxes(title_text="N", row=1, col=1)
fig.update_yaxes(title_text="相對次數", row=1, col=1)
fig.update_xaxes(title_text="log₁₀ 間隔（天）", row=1, col=2)
fig.update_yaxes(title_text="累積比例", row=1, col=2)
apply_layout(fig, barmode="overlay", height=430, hovermode="x",
             title=f"兩種模擬的一致性（各 {N_REP} 份目錄）："
                   f"平均 N＝{n_branch.mean():.1f} vs {n_thin.mean():.1f}，"
                   f"間隔分布最大差距 {d_max:.3f}")
fig

事件數分布重合，間隔分布的經驗累積曲線也幾乎疊在一起——兩條路確實產生同一
個過程。要說清楚的是這裡模擬的是**從空歷史出發、截斷在 $T$** 的過程（背景
事件只在 $[0,T]$ 內出生，後代超出 $T$ 就丟掉），兩邊的邊界處理一致，所以
不需要 burn-in 也能比。真正要做穩態模擬時，兩者都得往前多跑一段。

### 稀疏化的對偶：模擬與除叢是同一個運算

最後一個觀念是這一節真正的收尾。回頭看 thinning：我們用機率
$\lambda^*(s)/\bar\lambda$ 把**假的**提案點篩成真的目錄。現在把箭頭反過來
——給定一份**真的**目錄與一個擬合好的模型，每個事件 $j$ 是背景事件的機率是
$\phi_j = \mu/\lambda^*(t_j)$（分子是強度中「不是被誰觸發」的那一份）。用
這個機率把真實目錄篩一次，留下的就是一份背景目錄的實現——這就是**隨機除叢**
（stochastic declustering），第 14 章的主題。兩件事一模一樣：**都是拿一個
比值當機率去稀疏一組點**。模擬時稀疏假點，讓它們變成模型；除叢時稀疏真點，
讓它們變成背景。同一個運算，正反兩讀。

## 10.7 與其他學門的類比

地震學在這件事上並不孤單，而且往往不是最早或走得最遠的那一個。

**金融：訂單流與交易叢集**。高頻資料裡成交與掛單明顯成群出現——一筆大單
觸發一連串反應，跟餘震沒有兩樣。Hawkes 過程是市場微結構研究的標準工具，
用來量化「內生反饋佔比」：估出來的分支比若接近 1，代表市場活動絕大部分是
自己觸發自己，而不是回應外部資訊。這與地震學家說「一半以上的地震是餘震」
是同一句話、同一個參數。

**神經科學：spike train**。單個神經元的放電序列是最乾淨的點過程資料：有
不反應期（放電後強度歸零，self-correcting），也有爆發（burst，self-exciting）。
神經編碼研究的 point-process GLM 就是把 $\lambda^*$ 寫成「刺激濾波 + 自身
歷史濾波 + 其他神經元的歷史濾波」的指數，再用 {eq}`eq:pp-loglik` 估參數；
連檢驗方法都一樣——時間變換後畫 Q–Q 圖跑 KS 檢定，是神經生理論文的標配。

**流行病學：$R_0$ 與傳染鏈**。ETAS 的名字裡就有 epidemic，分支比在流行
病學裡叫基本再生數 $R_0$：大於 1 疫情擴散，小於 1 熄滅。COVID-19 期間有
大量研究直接用 Hawkes 過程建模病例序列，因為「一個病例平均感染幾個人、
間隔多久」與「一個地震平均觸發幾個地震、間隔多久」在數學上完全同構；
地震學的除叢問題，在流行病學裡叫「境外移入 vs 本土感染」。

**機器學習：神經點過程**。近年的一支做法是把 $\lambda^*$ 的函數形式整個
丟掉，改用 RNN 或 Transformer 把歷史編碼成一個向量、再由網路輸出強度
（Du et al. 2016；Zuo et al. 2020）。概似函數還是 {eq}`eq:pp-loglik`，
只是 $\lambda^*$ 換成了一個學出來的函數；這條路的潛力與代價，第 14 章的
前沿討論會再談。

這些類比不只是趣聞。它們的實際價值是：當你在地震學裡卡住時，別的領域可能
已經解過那個問題。

## 10.8 本部符號約定

地震統計文獻的符號是出了名的混亂：同一個字母在不同論文裡代表完全不同的
東西，還常常同時出現在同一個模型裡。這一節把第二部的約定攤開，往後每章
都照這張表寫。建議加入書籤——你會回來翻好幾次。

### 核心符號

| 概念 | 本書寫法 | 文獻中的其他寫法 | 備註 |
|---|---|---|---|
| 事件 | $(t_i,x_i,y_i,m_i)$ | $(t_i,\mathbf{x}_i,M_i)$ | 時間、位置、規模 |
| 歷史 | $H_t$ | $\mathcal{F}_t$、$\mathcal{H}_t$ | $t$ 之前的全部事件 |
| 條件強度 | $\lambda^*(t,x,y,m)$ | $\lambda(t\mid H_t)$、$\lambda_\theta$ | 星號＝條件於歷史 |
| compensator | $\Lambda(t)$ | $E_\lambda(N)$ | 積分後的期望數 |
| 網格期望數 | $\Lambda_{jk}$ | $\lambda_{jk}$ | 空間格 $j$ × 規模箱 $k$ |
| 網格觀測數 | $\omega_{jk}$ | $n_{jk}$ | 對應的實際計數 |

### 規模與觸發核

| 概念 | 本書寫法 | 文獻中的其他寫法 | 備註 |
|---|---|---|---|
| 目錄完整度 | $M_c$ | $m_c$、$M_{\rm comp}$ | 目錄的性質，是估計量 |
| 模型輸入門檻 | $m_0$ | $M_c$、$M_{\min}$ | 模型設定，通常 $m_0\ge M_c$ |
| 預報目標門檻 | $m_T$ | $M_{\rm target}$ | 目標地震集的下限 |
| GR 斜率 | $b$；$\beta=b\ln10$ | $\beta$ 有時另指他物 | $\beta$ **只**代表 $b\ln10$ |
| 規模密度 | $s(m)$ | $\nu_\beta(m)$、$g_0(m)$ | $\int s\,\mathrm{d}m=1$ |
| Omori 未正規化 | $n(t)=K(t+c)^{-p}$ | $\nu(t)$ | $K$ 帶量綱 |
| Omori 密度 | $g(t)$ | $g_{c,p}$、$f(t)$ | 積分為 1，需 $p>1$ |
| 產能 | $\kappa(m)=Ae^{\alpha(m-m_0)}$ | $Ke^{\alpha(M-M_c)}$ | 搭配正規化核 |
| 空間核 | $f(x,y;m)$ | $f_{D,\gamma,q}$ | 需 $q>1$ |
| 背景率 | $\mu(x,y)$ | $\tilde u(x,y)$、$\nu\mu(x,y)$ | 恆帶引數 |

### 三個最容易踩的地雷

第一，**$K$ 與 $A$ 不能直接比較**。Ogata 的 $n(t)=K(t+c)^{-p}$ 不是機率
密度，量綱藏在 $K$ 裡；本書的 $g$ 是密度（積分為 1），所有「量」都收進
$\kappa$。跨論文比較產能之前，先確認對方的時間核有沒有正規化、$m_0$ 是否
相同。第二，**$M_c$ 與 $m_0$ 是兩回事**：前者是目錄的性質（要估），後者是
你的選擇（要宣告）；很多論文兩者混用同一個符號，於是「$M_c$ 取 3.0」到底
是估出來的還是設定的，讀者無從判斷。第三，**$\lambda$ 與 $\Lambda$ 差一個
積分**：前者是率密度（帶量綱），後者是期望個數（無量綱）；CSEP 的網格化
預報交出來的是 $\Lambda_{jk}$，搞混會讓檢驗統計量差好幾個數量級。

### 必須改名的衝突符號

| 文獻用法 | 本書寫法 | 為什麼會撞 |
|---|---|---|
| 應力釋放模型的 $a,b,c$ | $a_s,b_s,c_s$ | 撞 GR 的 $b$、Omori 的 $c$ |
| BPT 平均複發時間 $\mu$ | $T_r$ | 撞 ETAS 背景率 $\mu$ |
| BPT aperiodicity $\alpha$ | $c_v$ | 撞 ETAS 產能指數 $\alpha$ |
| Weibull 形狀參數 $\beta$ | $k$（尺度用 $\theta$） | 撞 $\beta=b\ln10$ |
| EEPAS 混合權重 $\mu$ | $\mu_E$ | 撞背景率 $\mu$ |
| 各種機率 $p$ | 一律寫 $P(\cdot)$ | $p$ 保留給 Omori 指數 |

為什麼要這麼囉嗦？因為第二部後半會**同時**出現好幾個模型：第 19 章要把
兩類模型組合起來，第 20 章要把 BPT 與應力釋放放進同一個框架，第 22 章的
作業化系統裡三種模型並行。到那時候，如果 $\alpha$ 在同一頁上有三個意思，
讀者就再也跟不上了。統一符號不是潔癖，是為了讓後面幾章寫得出來。

## 10.9 常見誤解與陷阱

**一、把 $\lambda^*$ 當機率**。$\lambda^*$ 是率，單位是次／時間，可以是 50；
機率要透過 $P = 1-\exp(-\int\lambda^*)$ 算，永遠小於 1。常見的變形是「反正
$\lambda^*$ 很小，$P\approx\lambda^*\Delta t$ 就好」——這在短時間、低強度時
的確成立，但餘震序列剛開始時 $\lambda^*\Delta t$ 可以遠大於 1，近似完全
崩潰，還會算出「機率 3.7」這種東西。

**二、忽略 $-\int\lambda^*$ 那一項**。只看 $\sum\ln\lambda^*$ 比較模型，
等於獎勵「到處都喊高」。這一項還有兩個常見的細節錯誤：積分只積到最後一個
事件 $t_N$ 而不是觀測窗末端 $T$（漏掉「後來沒再發生」的資訊）；以及積分
區域縮到只包含有事件的地方。兩者都會讓 $\ln L$ 虛假地變好。

**三、以為 thinning 需要模型「可逆」或積得出來**。thinning 只需要兩樣
東西：算得出 $\lambda^*(s)$，以及一個有效的上界；它完全不需要 $\Lambda$ 的
解析形式或任何反函數。需要 cdf 反解得出來的是**反函數法**，那是用在抽單個
核（Omori、空間核）的時候。兩者常被混為一談。

**四、把殘差圖的偏離都當成物理訊號**。$\tau$–$j$ 圖上一段漂亮的寧靜，最
可能的解釋依序是：目錄不完整（例如大震後小地震被淹沒）、測站或定位程序
改變、規模尺度換算不一致、模型設定不當（例如把空間非均勻的背景率硬塞進
均勻假設），最後才是真的有物理訊號。在排除前四項之前主張第五項，是這個
領域反覆出現的失誤。

**五、把 self-exciting 讀成因果**。$\lambda^*$ 描述的是統計上的依賴：知道
A 發生了，B 的機率就上升；這不等於「A 用應力把 B 推出來」。兩個事件也可能
同被第三個因素驅動（慢滑事件、流體），統計上一樣呈現叢集。

**六、以為強度高就代表下一個地震會大**。在可分離假設下，規模完全由 $s(m)$
決定，與 $\lambda^*$ 無關。餘震序列裡強度很高，只是說「接下來會有很多
地震」；出現大事件的絕對機率確實跟著上升，但那是數量效應，不是規模分布
變了。

## 10.10 研究前沿與未解問題

**連續與網格化的張力**。本章的 $\lambda^*$ 是連續的率密度，而 CSEP 的檢驗
框架要求把預報交成網格化的期望數 $\Lambda_{jk}$。這個轉換不是無害的：格子
邊界會切斷叢集、解析度限制了模型能表達的空間結構，而且連續版與網格版的
$\ln L$ 在數學上不是同一個量，不能互相比較。網格化是為了讓不同模型能在
同一張桌子上被檢驗而付出的代價；有沒有既保持連續、又能公平比較的檢驗
協定，是 CSEP 社群仍在討論的問題（第 18 章）。

**可分離性何時失效**。10.4 節的分解漂亮得讓人不想放棄，但「規模與歷史
獨立」的證據其實不強。若大震前 $b$ 值真的系統性下降，$s$ 就依賴 $H_t$、
$\beta$ 不能分離，Aki 估計式也不再是 MLE。困難在於 $b$ 值的變化與目錄
完整度的變化幾乎無法區分（第 11 章），所以這個問題本質上卡在資料品質，
不是卡在理論。

**神經點過程的代價**。用網路取代 $\lambda^*$ 的函數形式，在資料量大的領域
表現很好；地震學的處境不同：目標事件（大地震）極少，而且我們要的往往不是
預測精度，而是**可解釋的參數**——分支比是多少？背景率的空間分布長怎樣？
一個黑箱的 $\lambda^*$ 給不出這些。比較有希望的方向是混合式：物理與經驗
結構打底，用網路學殘差。

**目錄的不確定性與同時事件**。整章都假設 $(t_i,x_i,y_i,m_i)$ 精確已知，
實際上位置誤差可達數公里、規模誤差 0.1–0.3，而且誤差隨時間變化；把測量
誤差傳播進點過程概似，理論上要對真實位置積分，計算量極大，多數研究乾脆
忽略，而忽略的後果沒有被系統性評估過。另一個相關的破口是**同時事件**：
大破裂可以在數秒內產生多個子事件，而目錄的時間解析度有限；當多筆事件被
記在同一個時刻，簡單點過程的假設破裂，$\ln L$ 的推導也跟著失效。這在
分鐘級的超短期預報裡是實際存在的問題。

## 10.11 附錄：本章推導細節

### A. 時空標記版概似的逐步展開

把 $[0,T]$ 切成寬 $\Delta t$ 的小段、把 $S\times[m_0,\infty)$ 切成小塊
（面積乘規模寬為 $\Delta v$），令 $\delta_{kl}\in\{0,1\}$ 標示小盒 $C_{kl}$
裡有沒有事件。每個小盒裡的事件數近似服從 Poisson，平均為
$\lambda^*(t_k,v_l)\,\Delta t\,\Delta v$，於是

$$\begin{aligned}
L &\approx \prod_{\delta_{kl}=1}
  \lambda^*(t_k,v_l)\,\Delta t\,\Delta v\,
  e^{-\lambda^*(t_k,v_l)\Delta t\Delta v}
  \prod_{\delta_{kl}=0} e^{-\lambda^*(t_k,v_l)\Delta t\Delta v} \\
  &= \left[\prod_{i=1}^{N}\lambda^*(t_i,x_i,y_i,m_i)\right]
     (\Delta t\Delta v)^N
     \exp\!\left[-\sum_{k,l}\lambda^*(t_k,v_l)\Delta t\Delta v\right]
\end{aligned}$$

取對數、令 $\Delta t,\Delta v\to0$，求和變積分；$(\Delta t\Delta v)^N$ 與
參數無關，做最大概似時可以丟掉（它只是把「機率」換算成「密度」的量綱
因子）。這條路線與 10.2 節的鏈式法則結果相同，但好處是**不需要「下一個
事件」的概念**，因此可以直接推廣到空間——空間上沒有「下一個」這種順序。

### B. 隨機時間變換：補完證明

10.5 節的證明只處理了單一個間隔。要得到「$\{\tau_i\}$ 是單位速率 Poisson
過程」，還需要三件事。（一）**$\Lambda$ 可逆**：要求 $\lambda^*>0$ 幾乎
處處成立，實務上總是滿足（背景率 $\mu>0$）；若模型允許 $\lambda^*=0$ 的
區段（某些 self-correcting 模型在應力極低時），$\Lambda$ 會有平台、反函數
不唯一，定理要改用廣義反函數陳述。（二）**$\Lambda(\infty)=\infty$**：
否則過程在有限的變換時間內就用完了，得到的是有限個點。（三）**遞迴地
套用**：條件於 $H_{t_i}$ 得到 $\Delta\tau_{i+1}\sim\mathrm{Exp}(1)$，而這個
分布**不依賴 $H_{t_i}$ 的內容**；既然條件分布與條件無關，就等於無條件分布，
且各間隔互相獨立。逐個 $i$ 套用，得到 i.i.d. 的 Exp(1) 間隔序列。

順帶說明 $U_j$ 的來歷：若 $X\sim\mathrm{Exp}(1)$，其 cdf 為 $F(x)=1-e^{-x}$，
則 $F(X)\sim U(0,1)$（機率積分變換，與 10.6 節的反函數法是同一條定理的
正反兩面）。$U_j=1-e^{-\Delta\tau_j}$ 把指數間隔攤平成均勻分布，讓 Q–Q 圖
畫在一個有界的方框裡，比直接畫指數分布好讀。

### C. 為什麼加權殘差的期望是零

10.5 節的 $R(I\times B;h)$ 之所以能當殘差用，靠的是一條 Campbell 型的
等式：對任何**可預測的**（predictable，也就是 $h(t)$ 只依賴 $t$ 之前的
資訊）權重函數，

$$E\left[\sum_{i} h(t_i,x_i,y_i)\right]
  = E\left[\iiint h(t,x,y)\,\lambda^*(t,x,y)\,
    \mathrm{d}x\,\mathrm{d}y\,\mathrm{d}t\right]$$

直觀的理由就是條件強度的定義：在小盒 $\mathrm{d}t\,\mathrm{d}A$ 裡「有
事件」的條件機率是 $\lambda^*\,\mathrm{d}t\,\mathrm{d}A$，所以每個小盒對
左式的期望貢獻是 $h\cdot\lambda^*\,\mathrm{d}t\,\mathrm{d}A$，加總即右式；
兩邊相減就得到 $E[R]=0$。「可預測」不是形式主義：$h=1/\lambda^*$ 與
$h=1/\sqrt{\lambda^*}$ 都合格，因為 $\lambda^*(t)$ 只用到 $t$ 之前的歷史；
但若手滑用了會偷看未來的權重（例如「這個事件後來有沒有餘震」），等式就
不成立，殘差圖會出現憑空的系統性偏差。

### D. 反函數法在實作上的兩個細節

第一，程式裡常寫 `1 - U` 而不是 `U`，雖然兩者同分布：多數亂數產生器回傳
$[0,1)$，`U` 可能剛好是 0，而 $0^{-1/(p-1)}$ 會溢位，`1 - U` 則落在
$(0,1]$。第二，Omori 核有**極重的尾巴**（$p$ 接近 1 時尤其），抽出來的
$\Delta t$ 偶爾會大得離譜——這不是 bug，是冪次分布的正常行為；模擬時務必
先檢查 $t_i+\Delta t$ 有沒有超出時間窗再決定要不要保留。

## 這一章是語言，不是模型

回頭看，這一章沒有介紹任何一個地震模型。沒有 ETAS 的參數表，沒有台灣的
目錄，沒有一張真實資料的圖；所有的圖都是合成的，所有的數字都是模擬跑出
來的。這是刻意的。點過程是**語法**，不是句子。$\lambda^*$ 怎麼寫是模型的
事，但只要寫下來了，剩下的一切就都是自動的：機率是 $1-\exp(-\int\lambda^*)$，
概似是 $\sum\ln\lambda^*-\int\lambda^*$，檢驗是把時間按 $\Lambda$ 拉直看
像不像 Poisson，模擬是 thinning 或 branching。**你不需要為每個新模型重新
發明這些東西**。這種「一次學會、處處適用」的槓桿在科學裡不常見，遇到了
就該用力握住。接下來十三章會出現十幾個模型，它們的物理動機南轅北轍——有的
講觸發、有的講前兆、有的講應力耗竭——但全部都在填同一個空格：$\lambda^*$
等於什麼。你讀每一個新模型時，第一個該問的問題永遠是那一句。

語言就緒了，接下來要處理材料。點過程吃的是地震目錄，而地震目錄從來不是
乾淨的：它有偵測門檻、有規模尺度的混用、有隨時間變動的完整度。這些不是
細節，它們會直接汙染上面每一條式子——$m_0$ 選錯，$\hat\beta$ 就偏；完整度
隨時間漂移，$\tau$–$j$ 圖就長出假的寧靜。
{doc}`第 11 章 <11_catalog_completeness_b>`要把目錄本身攤開來檢查，
從 GR 律與 $b$ 值開始。